# Vanilla Model Scoring (GPT-4o / GPT-4o-mini)

This notebook scores vanilla (prompted) GPT models and writes logprobs CSVs.


Related notebooks:
- `LogProbsOcsai1.ipynb` (trained model scoring)
- `LogProbsAnalysis.ipynb` (analysis and figures)


In [3]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()  # loads OPENAI_API_KEY from .env file if present

True

In [4]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
import numpy as np

tqdm.pandas()

data_dir = Path('../../data')

In [5]:
# load original English uses data
datasets = {}
for split in ['test', 'train']:
    datasets[split] = pd.read_json(data_dir/ 'ocsai1'/ f'finetune-gt_main2_prepared_{split}.jsonl', lines=True)
    def parse_prompt(p):
        parts = p.split('\n')
        prompt = parts[0].split(':', 1)[1]
        response = parts[1].split(':', 1)[1]
        return pd.Series({'prompt': prompt, 'response': response})
    datasets[split][['item', 'response']] = datasets[split]['prompt'].apply(parse_prompt)
    datasets[split]['target'] = pd.to_numeric(datasets[split]['completion']).div(10)
    datasets[split]['type'] = 'uses'
    datasets[split].item.value_counts()

data = datasets['test']

# Score

## Score Ocsai 1 Uses, with Plain (Prompted) GPT-4o

In [6]:
# Create a basic few shot prompt by sorting, taking every nth example, then randomizing
fewshot_sample = datasets['train'].sort_values('completion', ascending=False).iloc[::150].sample(frac=1, random_state=42)
display(fewshot_sample.iloc[0])
print("# examples:", len(fewshot_sample))

import tiktoken
encoding = tiktoken.encoding_for_model('gpt-4o')
example_str = ''
for row in fewshot_sample.prompt.str.strip() + fewshot_sample.completion.astype(str):
    example_str += f"\n{row}\n"

print("# tokens:", len(encoding.encode(example_str)))
print('\n```' + example_str[:140] + '...\n```')


prompt        AUT Prompt:hat\nResponse:put it backwards on y...
completion                                                   16
item                                                        hat
response                         put it backwards on your head.
target                                                      1.6
type                                                       uses
Name: 9528, dtype: object

# examples: 108
# tokens: 1924

```
AUT Prompt:hat
Response:put it backwards on your head.
Score:16

AUT Prompt:fork
Response:pick my nose
Score:35

AUT Prompt:ball
Response:c...
```


In [7]:
from ocsai.prompter.ocsai1_chat_prompter import Ocsai1_Chat_Prompter
from ocsai.types import FullScore, ResponseTypes
import textwrap

class GPT_Vanilla_Prompter(Ocsai1_Chat_Prompter):
    sys_msg_text = textwrap.dedent('''
    You score originality in the alternate uses task.

    On a scale of 10-50, judge how original each use is, for the give item, where 10 is 'not at all creative' and 50 is 'very creative'
                                   
    Respond with *JUST* a single number, no other text, just one token score.
                                   
    Here are examples of the task:
    ''' + '\n' + example_str)

    def craft_prompt(self, item, response, task_type='uses', question=None, language='eng'):
        ''' Remove "\nScore:\n" from the legacy format.'''
        prompt = super().craft_prompt(item, response, task_type, question, language)
        return prompt + '\nScore:'

In [8]:
from ocsai.inference import Chat_Scorer
chatscorer = Chat_Scorer(prompter=GPT_Vanilla_Prompter(),
                             model_dict={'gpt-4o':'gpt-4o-2024-08-06',
                                         'gpt-4o-mini':'gpt-4o-mini-2024-07-18',
                                         'gpt-4-1':'gpt-4.1-2025-04-14',
                                         'gpt-4-1-mini':'gpt-4.1-mini-2025-04-14'
                                         })
                                         # reasoning models like o3-mini and gpt-5 don't allow requesting logprobs;
                                         # a way to get around this is to generate the '<think>' response, then use 4.1 to score,
                                         # after the thought trace is in the response.
                                         # - but the paper is already way to0 big to go in-depth here.
                                         # Here is your study idea, reader :)

In [10]:
def score_weighted(row, model='gpt-4o', top_probs=5, scorer=chatscorer):
    cols = dict()
    try:
        scores = scorer.score(row['item'],
                            response=row['response'], 
                            task_type=row['type'],
                            model=model,
                            top_probs=top_probs,
                            progressive_weighted=True)
        scores = sorted(scores, key=lambda x: x['n'])
        
        for s in scores:
            cols[f'score_{s["n"]}'] = s['score']
            cols[f'confidence_{s["n"]}'] = s['confidence']
    except KeyboardInterrupt:
        raise
    except:
        print(f"Problem with {row['prompt']}")
        for s in range(1, top_probs+1):
            cols[f'score_{s}'] = None
            cols[f'confidence_{s}'] = None
        raise
    return pd.Series(cols)

print(data.iloc[0])
score_weighted(data.iloc[0], top_probs=20)

prompt        AUT Prompt:paperclip\nResponse:Paperclip jewel...
completion                                                   20
item                                                  paperclip
response                                      Paperclip jewelry
target                                                      2.0
type                                                       uses
Name: 0, dtype: object


score_1          2.800000
confidence_1     0.244643
score_2          2.840733
confidence_2     0.412783
score_3          2.782757
confidence_3     0.543730
score_4          2.820836
confidence_4     0.659291
score_5          2.804648
confidence_5     0.761273
score_6          2.784265
confidence_6     0.815861
score_7          2.810336
confidence_7     0.870448
score_8          2.822305
confidence_8     0.907965
score_9          2.813167
confidence_9     0.928046
score_10         2.820301
confidence_10    0.941848
score_11         2.828673
confidence_11    0.955650
score_12         2.837122
confidence_12    0.967830
score_13         2.832516
confidence_13    0.976201
score_14         2.827765
confidence_14    0.983589
score_15         2.831267
confidence_15    0.988069
score_16         2.828987
confidence_16    0.990787
score_17         2.827433
confidence_17    0.992904
score_18         2.829289
confidence_18    0.995020
score_19         2.830393
confidence_19    0.996153
score_20    

In [ ]:
# test run, to check that everything is oll korrect
test = data.sample(100, random_state=42)
top_probs = 20
model = 'gpt-4-1'
scorer = chatscorer

responses = test.progress_apply(lambda row: score_weighted(row, model=model, top_probs=top_probs, scorer=scorer), axis=1)
test[responses.columns] = responses.values
test

In [11]:
#test = data.sample(100)
top_probs = 20
model = 'gpt-4-1-mini'
scorer = chatscorer
responses = data.progress_apply(lambda row: score_weighted(row, model=model, top_probs=top_probs, scorer=scorer), axis=1)
data[responses.columns] = responses.values

  0%|          | 0/3030 [00:00<?, ?it/s]

In [179]:
for i in range(1, 20):
    # get correlation between score_i and target
    ignore = data[f'score_{i}'].isna()
    if ignore.sum() > 0:
        print(f'Score {i} has {ignore.sum()} missing values')
    
    # Calculate both correlation and RMSE for non-missing values
    valid_data = data[~ignore]
    r = pearsonr(valid_data[f'score_{i}'], valid_data['target'])[0]
    rmse = np.sqrt(mean_squared_error(valid_data['target'], valid_data[f'score_{i}']))
    print(f'Score {i} - correlation: {r:.3f}, RMSE: {rmse:.4f}')

Score 1 - correlation: 0.607, RMSE: 0.7656
Score 2 - correlation: 0.618, RMSE: 0.7497
Score 3 - correlation: 0.620, RMSE: 0.7435
Score 4 - correlation: 0.622, RMSE: 0.7393
Score 5 - correlation: 0.622, RMSE: 0.7376
Score 6 - correlation: 0.623, RMSE: 0.7348
Score 7 - correlation: 0.624, RMSE: 0.7326
Score 8 - correlation: 0.624, RMSE: 0.7313
Score 9 - correlation: 0.625, RMSE: 0.7302
Score 10 - correlation: 0.625, RMSE: 0.7287
Score 11 - correlation: 0.626, RMSE: 0.7283
Score 12 - correlation: 0.626, RMSE: 0.7267
Score 13 - correlation: 0.627, RMSE: 0.7259
Score 14 - correlation: 0.627, RMSE: 0.7256
Score 15 - correlation: 0.627, RMSE: 0.7253
Score 16 has 6 missing values
Score 16 - correlation: 0.625, RMSE: 0.7255
Score 17 has 13 missing values
Score 17 - correlation: 0.624, RMSE: 0.7263
Score 18 has 27 missing values
Score 18 - correlation: 0.620, RMSE: 0.7275
Score 19 has 55 missing values
Score 19 - correlation: 0.614, RMSE: 0.7297


In [13]:
data.to_csv(data_dir / 'results/logprobs' / f'logprobs_{model}_ocsai1.csv', index=False)

### Example Probs, for paper

np.exp(log_prob)

In [248]:
example1 = data.loc[1989].copy()
example1['score_10'] = np.round(example1['score_10'], 1)
example1[['item', 'response', 'target', 'score_1', 'score_10']]

item            paperclip
response    remove grout 
target                2.5
score_1               2.2
score_10              2.5
Name: 1989, dtype: object

In [262]:
#score_weighted(example1, top_probs=10) # ugly - I printed the probs_to_parse[-1] inside the base_scorer and copied here
logprobs = [('22', -1.85019), ('26', -1.85019), ('28', -2.2251902), ('24', -2.3501902), ('29', -2.7251902), ('23', -2.7251902), ('27', -2.8501902), ('20', -2.9751902), ('21', -2.9751902), ('25', -3.1001902)]
for score, logprob in logprobs:
    print("token:", int(score)/10, "prob:", (np.exp(logprob) * 100).round(2))

token: 2.2 prob: 15.72
token: 2.6 prob: 15.72
token: 2.8 prob: 10.8
token: 2.4 prob: 9.54
token: 2.9 prob: 6.55
token: 2.3 prob: 6.55
token: 2.7 prob: 5.78
token: 2.0 prob: 5.1
token: 2.1 prob: 5.1
token: 2.5 prob: 4.5
